In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# Real-Time Patient Monitoring
# ========================================

from pyspark.sql.functions import *

# Read Streaming ICU Data
stream_df = (
    spark.readStream
    .format("csv")
    .option("header", True)
    .schema("""
        patient_id STRING,
        heart_rate INT,
        oxygen_level INT,
        body_temperature DOUBLE,
        event_time TIMESTAMP
    """)
    .load(f"{source_path}/icu_stream")
)

# Identify Critical Patients
critical_alert_df = stream_df.filter(
    (col("heart_rate") > 120) |
    (col("oxygen_level") < 90) |
    (col("body_temperature") > 102)
)

# Write Alerts to Gold Layer
query = (
    critical_alert_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(once=True)
    .option(
        "checkpointLocation",
        f"{checkpoint_path}/icu_patient_alerts"
    )
    .start(f"{gold_path}/icu_patient_alerts")
)

alerts_df = spark.read.format("delta").load(
    f"{gold_path}/icu_patient_alerts"
)

display(alerts_df)

query.awaitTermination()